# Comparing districting plans

<style>
blockquote:has(.notebook-admonition-title) {
  --notebook-admonition-color: var(--color-admonition-title--note, #087fc7);
  --notebook-admonition-title-background:
    var(--color-admonition-title-background--note, rgba(8, 127, 199, 0.18));
  background: var(--color-admonition-background, transparent);
  border: 0;
  border-left: 0.2rem solid var(--notebook-admonition-color);
  border-radius: 0.2rem;
  box-shadow: 0 0.2rem 0.5rem rgba(0, 0, 0, 0.05), 0 0 0.0625rem rgba(0, 0, 0, 0.1);
  font-size: var(--admonition-font-size, 0.8125rem);
  margin: 1rem auto;
  overflow: hidden;
  padding: 0 0.5rem 0.5rem;
}
blockquote p:has(> .notebook-admonition-title) {
  background: var(--notebook-admonition-title-background);
  font-size: var(--admonition-title-font-size, 0.8125rem);
  font-weight: 500;
  line-height: 1.3;
  margin: 0 -0.5rem 0.5rem;
  padding: 0.4rem 0.5rem 0.4rem 2rem;
  position: relative;
}
blockquote p:has(> .notebook-admonition-title)::before {
  color: var(--notebook-admonition-color);
  content: "✎";
  left: 0.65rem;
  position: absolute;
}
.notebook-admonition-title {
  font-weight: inherit;
}
table:not(.dataframe) {
  border: 1px solid var(--docs-hairline, rgba(128, 128, 128, 0.35));
  border-collapse: collapse;
}
table:not(.dataframe) th,
table:not(.dataframe) td {
  border: 1px solid var(--docs-hairline, rgba(128, 128, 128, 0.35));
}
</style>

<div style="text-align: center;"><a class="sd-sphinx-override sd-btn sd-text-wrap sd-btn-primary reference external" href="https://github.com/mggg/gerrytools/tree/main/user_guide/_static/data">Browse tutorial data</a></div>

Georgia enacted a congressional plan after the 2020 Census and replaced it with a
remedial plan in 2023. This guide compares the two plans by census-block population and
by district geometry. The main question is not whether identically numbered districts
overlap. It is how much of the 2021 plan the 2023 plan retains after the district labels
are put on comparable footing.

The example combines 2020 Census P1 population with the Census Bureau's block-equivalency
files for the [118th Congress][cd118] and
[119th Congress][cd119]. Those assignments correspond to Georgia's
[2021 SB 2EX][sb2ex] and
[2023 SB 3EX][sb3ex] plans. The district geometries are dissolved from the same block
assignments and projected to EPSG:5070.

[cd118]: https://www.census.gov/geographies/mapping-files/2023/dec/rdo/118-congressional-district-bef.html
[cd119]: https://www.census.gov/geographies/mapping-files/2025/dec/rdo/119-congressional-district-bef.html
[sb2ex]: https://gov.georgia.gov/document/2021-special-session-signed-legislation/sb-2ex/download
[sb3ex]: https://gov.georgia.gov/document/2023-special-session-signed-legislation/sb-3ex/download


In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

from gerrytools.plan_comparison import (
    areal_overlap,
    minimum_population_dispersion,
    optimal_relabeling,
    population_dispersion_by_district,
    population_overlap,
)
from gerrytools.plotting import GeoPlot

blocks = pd.read_csv(
    "data/ga_congressional_plans.csv.gz",
    dtype={"GEOID20": "string", "CD_2021": "string", "CD_2023": "string"},
)
districts = gpd.read_file(
    "data/ga_congressional_plans.gpkg",
    layer="districts",
)

districts_21 = districts.query("PLAN == '2021'").copy()
districts_23 = districts.query("PLAN == '2023'").copy()

blocks.head()

Each block row contains total population and both district assignments. The geometry file
has one polygon per district and plan. Drawing the plans before comparing them makes one
complication visible: several district numbers move with the redraw, so matching the raw
labels would overstate the change.


In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(16, 8))
gp = GeoPlot(districts_21, title="2021 plan")
gp.add_districting_plan_layer("DISTRICT", show_labels=True, label_style="halo")
gp.bind_to_ax(axes[0])

gp = GeoPlot(districts_23, title="2023 plan")
gp.add_districting_plan_layer("DISTRICT", show_labels=True, label_style="halo")
gp.bind_to_ax(axes[1])


plt.show()

## Population overlap

`population_overlap()` aggregates atomic units into a matrix. Rows are districts in the
source plan, columns are districts in the target plan, and each cell is the population
shared by that pair. Here the 2023 plan is the source to be relabeled and the 2021 plan is
the reference target.

The table is shown in millions of people. Districts 1, 2, 3, 8, and 12 remain on the
diagonal, while the lower-right portion records most of the redraw.


In [ ]:
overlap = population_overlap(
    blocks,
    source="CD_2023",
    target="CD_2021",
    population="TOTPOP20",
)

(overlap.sort_index().sort_index(axis=1) / 1_000_000).round(2)

## Optimize the district labels

District labels are names, not measurements. `optimal_relabeling()` finds the one-to-one
source-to-target assignment that retains the greatest overlap. It solves a linear
assignment problem over the overlap matrix; it does not alter either plan.


In [ ]:
relabeling = optimal_relabeling(overlap)

(pd.Series(relabeling, name="matched 2021 label").sort_index().rename_axis("2023 label").to_frame())

Applying the optimized labels to the 2023 plan makes the colors comparable across the two
maps. The district geometries do not change; only the labels used to assign their colors do.


In [ ]:
districts_23["DISTRICT_matched"] = districts_23["DISTRICT"].map(relabeling)

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(16, 8))
gp = GeoPlot(districts_21, title="2021 plan")
gp.add_districting_plan_layer("DISTRICT", show_labels=True, label_style="halo")
gp.bind_to_ax(axes[0])

gp = GeoPlot(districts_23, title="2023 plan (after matching)")
gp.add_districting_plan_layer("DISTRICT_matched", show_labels=True, label_style="halo")
gp.bind_to_ax(axes[1])


plt.show()

In this comparison, 2023 districts 6, 7, and 13 match 2021 districts 13, 6, and 7. The
remaining labels already maximize their own population overlap.

`minimum_population_dispersion()` performs the overlap calculation and label optimization
together. Its `population` field is the number of people assigned outside their matched
2021 district, not the population in districts whose boundaries changed at all.


In [ ]:
comparison = minimum_population_dispersion(
    blocks,
    reference="CD_2021",
    comparison="CD_2023",
    population="TOTPOP20",
)

pd.Series(
    {
        "reassigned population": f"{comparison.population:,.0f}",
        "share of Georgia population": (f"{comparison.population / blocks['TOTPOP20'].sum():.1%}"),
    }
)

Statewide dispersion compresses the comparison to one number. To locate that movement,
first apply the optimized labels to the 2023 assignment and then call
`population_dispersion_by_district()`. The function groups reassigned population by the
reference district, so the denominator below is each 2021 district's population.


In [ ]:
blocks["CD_2023_MATCHED"] = blocks["CD_2023"].map(comparison.relabeling)

reassigned = population_dispersion_by_district(
    blocks,
    reference="CD_2021",
    comparison="CD_2023_MATCHED",
    population="TOTPOP20",
)
reference_population = blocks.groupby("CD_2021")["TOTPOP20"].sum()
by_district = pd.DataFrame(
    {
        "population": reference_population,
        "reassigned population": reassigned,
        "reassigned share": reassigned / reference_population,
    }
).sort_values("reassigned share", ascending=False)

by_district

The table separates the untouched districts from the centers of the redraw. More than half
of 2021 district 7's population and nearly half of district 13's population fall outside
their matched 2023 districts.

## Compare plans by area

`areal_overlap()` constructs the same source-target matrix from district intersections.
It requires one valid geometry per district and a projected CRS. Population and area
answer different questions, so their optimal labelings can differ. For these Georgia
plans, both objectives select the same mapping.


In [ ]:
plan_2021 = districts.loc[districts["PLAN"].eq("2021"), ["DISTRICT", "geometry"]]
plan_2023 = districts.loc[districts["PLAN"].eq("2023"), ["DISTRICT", "geometry"]]

area_overlap = areal_overlap(
    plan_2023,
    plan_2021,
    source_label="DISTRICT",
    target_label="DISTRICT",
)
area_relabeling = optimal_relabeling(area_overlap)

pd.DataFrame(
    {
        "population match": pd.Series(comparison.relabeling),
        "area match": pd.Series(area_relabeling),
    }
).sort_index().rename_axis("2023 label")

> <span class="notebook-admonition-title">Advanced: parity-constrained relabeling</span>
>
> `minimum_population_dispersion_with_parity()` is not a generic partisan or
> parity-of-population constraint. It implements the Wisconsin staggered state-senate
> convention: first assign the required number of even reference labels to minimize the
> population shifted from odd reference districts, then maximize retained population
> without worsening that result. Georgia congressional districts have no corresponding
> constraint, so applying it to this example would give the labels a meaning they do not
> have. See the [plan-comparison API](../api/plan_comparison.rst) for the full formulation.

## Related

- [Geographic plotting](plotting/geographic/workflow.ipynb) covers the map-building API.
- [Plan comparison API](../api/plan_comparison.rst) documents every overlap and dispersion
  function.
